In [1]:
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt

from gensim.models import Word2Vec
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
# --------------------------------------------------
# STEP 1: LOAD AND PREPROCESS DATASET
# --------------------------------------------------

data = pd.read_csv("bbc-text.csv")

# Remove missing values
data = data.dropna(subset=['text'])

# Store article text and category
documents = data['text'].astype(str).tolist()
categories = data['category'].tolist()

print("Total Articles:", len(documents))

# Text preprocessing function
def preprocess_text(text):

    text = text.lower()

    # Remove special characters and numbers
    text = re.sub(r'[^a-z\s]', ' ', text)

    # Tokenize text
    words = text.split()

    # Remove stop words
    words = [
        word for word in words
        if word not in ENGLISH_STOP_WORDS
    ]

    return words


# Prepare all documents
train_sentences = [
    preprocess_text(doc)
    for doc in documents
]

print("\nData Preprocessing Completed")

Total Articles: 2225

Data Preprocessing Completed


In [3]:
# --------------------------------------------------
# STEP 2: TRAIN WORD2VEC CBOW MODEL
# --------------------------------------------------

model = Word2Vec(
    sentences=train_sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=0,
    epochs=10
)

print("\nCBOW Model Trained Successfully")


CBOW Model Trained Successfully


In [4]:
# --------------------------------------------------
# STEP 3: CREATE DOCUMENT VECTORS
# --------------------------------------------------

def document_vector(words):

    vectors = [
        model.wv[word]
        for word in words
        if word in model.wv
    ]

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)


# Convert every article into a vector
document_vectors = np.array([
    document_vector(doc)
    for doc in train_sentences
])

print("\nDocument Vectors Created")
print("Document Vector Shape:", document_vectors.shape)


Document Vectors Created
Document Vector Shape: (2225, 100)


In [5]:
# --------------------------------------------------
# STEP 4: RECOMMENDATION FUNCTION
# --------------------------------------------------

def recommend_articles(user_input, top_n=5):

    # Preprocess user input
    input_words = preprocess_text(user_input)

    # Convert user input into a vector
    input_vector = document_vector(input_words)

    # Calculate cosine similarity
    similarities = cosine_similarity(
        [input_vector],
        document_vectors
    )[0]

    # Get most similar articles
    top_indices = np.argsort(similarities)[::-1]

    print("\nRecommended Articles:\n")

    count = 0

    for idx in top_indices:

        # Skip articles with no similarity
        if similarities[idx] <= 0:
            continue

        print("Article:", count + 1)

        print("Category:", categories[idx])

        print("Similarity Score:", round(similarities[idx], 4))

        print("Content:", documents[idx][:300], "...")

        print("-" * 60)

        count += 1

        if count == top_n:
            break

    if count == 0:
        print("No similar articles found.")


In [6]:
# --------------------------------------------------
# STEP 5: TEST RECOMMENDATION SYSTEM
# --------------------------------------------------

user_input = input("Enter your search topic or keywords: ")

recommend_articles(user_input, top_n=5)


Enter your search topic or keywords:  Artificial



Recommended Articles:

Article: 1
Category: politics
Similarity Score: 0.9622
Content: parliament s record of scandal in a locked room at the heart of parliament there is a hive of scandal.  sex  betrayal and custody of children are all there in this affair but this time it has nothing to do with the recent troubles of david blunkett or boris johnson.  few realise that westminster in  ...
------------------------------------------------------------
Article: 2
Category: tech
Similarity Score: 0.9507
Content: tech helps disabled speed demons an organisation has been launched to encourage disabled people to get involved in all aspects of motorsport  which is now increasingly possible thanks to technological innovations.  the motorsport endeavour club left the starting grid yesterday at the autosport inter ...
------------------------------------------------------------
Article: 3
Category: tech
Similarity Score: 0.9491
Content: ink helps drive democracy in asia the kyrgyz republic  a sma

In [7]:
# --------------------------------------------------
# STEP 6: SAVE CBOW MODEL
# --------------------------------------------------

model.save("recommendation_cbow.model")

print("\nCBOW Model Saved Successfully")



CBOW Model Saved Successfully
